# A database of topics: one folder per topic, one numpy array each

`library()` is a folder. Every topic inside it is its own small store. Questions fan out:
**embed the prompt once, scan every topic, merge by score.** Archiving a topic is moving its folder.

That is the whole database. No server, no schema, nothing to migrate.

Needs the **Python 3 (trading)** kernel and Ollama with `nomic-embed-text`. Cells can be re-run in any order.

In [1]:
import sys; from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from slim_llm_memory import library

db = library(ROOT / ".library_nb")

db.topic("slim-llm-memory").add([ROOT / "README.md", ROOT / "docs" / "IMPLEMENTATION.md"])
db.topic("obsidian brain").add(ROOT / "docs" / "specs")
db.topic("cooking").add({
    "pasta.md": "Boil 100 g pasta per person in salted water. Save a cup of the water before draining; "
                "it binds the sauce.\n\nFor a quick sauce, fry garlic in olive oil, add the pasta and a "
                "splash of the pasta water.",
    "bread.md": "No-knead bread: 400 g flour, 300 g water, 8 g salt, 1 g yeast. Mix, rest 12 hours, "
                "shape, bake 45 minutes in a covered pot at 230 °C.",
})
db

library(/home/trbck/workspace/slim-llm-memory/.library_nb, ollama:nomic-embed-text): 3 active, 0 archived
  cooking             2 doc(s)       2 chunks  active
  obsidian brain      1 doc(s)      21 chunks  active
  slim-llm-memory     2 doc(s)      38 chunks  active

## Ask the whole library

One embedding call, then one numpy scan per topic. Hits are labelled with their topic and merged by score.

In [2]:
r = db.ask("what happens if the process crashes during a flush?", k=3)
r

ask('what happens if the process crashes during a flush?')  3 hit(s) · embed 2471 ms · scan 0.33 ms
   1  0.60  obsidian brain/2026-05-20-obsidian-brain-design.md#17 | Failure | Behaviour | |---|---| | Ollama unreachable | `Memory.upser
   2  0.57  slim-llm-memory/IMPLEMENTATION.md#10 Acceptance: - `Memory.upsert([…])` of 1000 items completes in <30 s on
   3  0.56  obsidian brain/2026-05-20-obsidian-brain-design.md#4 **Why two processes, not one thread:** the MCP server's lifecycle is o

In [3]:
r.per_topic_ms      # the scan cost, per topic, in milliseconds

{}

In [4]:
db.ask("what can I cook tonight without kneading?", k=2)

ask('what can I cook tonight without kneading?')  2 hit(s) · embed 1852 ms · scan 0.12 ms
   1  0.66  cooking/bread.md#0       No-knead bread: 400 g flour, 300 g water, 8 g salt, 1 g yeast. Mix, re
   2  0.50  cooking/pasta.md#0       Boil 100 g pasta per person in salted water. Save a cup of the water b

Restrict a question to some topics when you already know where the answer lives:

In [5]:
db.ask("how are notes chunked?", k=2, topics=["obsidian brain"])

ask('how are notes chunked?')  2 hit(s) · embed 1906 ms · scan 0.17 ms
   1  0.62  obsidian brain/2026-05-20-obsidian-brain-design.md#7 Edge cases the parser handles: - YAML frontmatter that's invalid → par
   2  0.53  obsidian brain/2026-05-20-obsidian-brain-design.md#5 ``` ~/Vault/ # the user's existing Obsidian vault Daily/ Projects/ Peo

## Two calls: find the topic, then search it

Each topic has a centroid (the mean of its vectors). `route` ranks topics against the prompt with one
tiny matrix product. `ask(..., route=True)` does that first and scans only the chosen topics.
With three topics this is a curiosity; with hundreds it cuts the scan 10–20× (see `examples/03_routing_bench.py`).
By default `ask` stays exact until the library passes 50k chunks, then routes on its own.

In [6]:
db.route("how do I bake bread without kneading?")

route('how do I bake bread without kneading?')  → ['cooking']  · embed 2151 ms · route 0.32 ms
  0.69  cooking
  0.45  obsidian brain
  0.42  slim-llm-memory

In [7]:
db.ask("how do I bake bread without kneading?", k=2, route=True)

ask('how do I bake bread without kneading?')  2 hit(s) · embed 1024 ms · scan 0.11 ms · routed to 1/3 topics in 0.08 ms
   1  0.79  cooking/bread.md#0       No-knead bread: 400 g flour, 300 g water, 8 g salt, 1 g yeast. Mix, re
   2  0.43  cooking/pasta.md#0       Boil 100 g pasta per person in salted water. Save a cup of the water b

## Archive a topic

`archive` moves the folder under `_archive/`. It drops out of `ask()` but stays on disk, searchable on request.

In [8]:
db.archive("cooking")
db

library(/home/trbck/workspace/slim-llm-memory/.library_nb, ollama:nomic-embed-text): 2 active, 1 archived
  obsidian brain      1 doc(s)      21 chunks  active
  slim-llm-memory     2 doc(s)      38 chunks  active
  cooking             2 doc(s)       2 chunks  archived

In [9]:
print("default:         ", [h.meta["topic"] for h in db.ask("pasta water", k=2)])
print("include_archived:", [h.meta["topic"] for h in db.ask("pasta water", k=2, include_archived=True)])

default:          ['slim-llm-memory', 'slim-llm-memory']


include_archived: ['cooking', 'slim-llm-memory']


In [10]:
db.restore("cooking")
db.topics()

[cooking: 2 doc(s), 2 chunks,
 obsidian brain: 1 doc(s), 21 chunks,
 slim-llm-memory: 2 doc(s), 38 chunks]

## The database on disk

Each topic is a directory: the texts as JSONL, the vectors as one `.npy`, and a manifest that makes writes atomic.
Back it up with `cp -r`, move it with `mv`, inspect it with `cat`.

In [11]:
for p in sorted(db.path.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(db.path).as_posix():<50} {p.stat().st_size:>9,} B")

cooking/.lock                                              0 B
cooking/items.v1.jsonl                                   615 B
cooking/manifest.json                                    146 B
cooking/topic.json                                        50 B
cooking/vectors.v1.npy                                 6,272 B
obsidian-brain/.lock                                       0 B
obsidian-brain/items.v1.jsonl                         21,242 B
obsidian-brain/manifest.json                             148 B
obsidian-brain/topic.json                                 56 B
obsidian-brain/vectors.v1.npy                         64,640 B
slim-llm-memory/.lock                                      0 B
slim-llm-memory/items.v1.jsonl                        34,445 B
slim-llm-memory/manifest.json                            148 B
slim-llm-memory/topic.json                                56 B
slim-llm-memory/vectors.v1.npy                       116,864 B


## Takeaways

- **Extending the database is `db.topic("new name").add(...)`.** A topic is a folder; no registration step.
- **Fan-out stays fast.** The prompt is embedded once; each topic costs one matrix-vector product.
- **Archive is a move, delete is an rm.** The filesystem is the schema.

In [12]:
db.close()